# KL Divergence — Interactive Notebook

**Concept 38** of the math-foundations learning map. Stdlib only — runs anywhere.

We explore the Kullback–Leibler divergence
$$D_{\mathrm{KL}}(p \,\|\, q) = \mathbb{E}_p\!\left[\log \frac{p(X)}{q(X)}\right]$$
and verify its key properties numerically: non-negativity, asymmetry, failure of the triangle inequality, and the closed-form Gaussian expression.


## 1. Imports and a discrete KL helper

We use only `math` and `random`. The helper handles the conventions $0\log 0 = 0$ and returns $+\infty$ if $q$ has a zero where $p$ does not.


In [ ]:
from math import log, pi, sqrt
import random

def kl_discrete(p, q):
    total = 0.0
    for pi_, qi in zip(p, q):
        if pi_ == 0.0:
            continue
        if qi == 0.0:
            return float('inf')
        total += pi_ * log(pi_ / qi)
    return total

kl_discrete([0.5, 0.5], [0.5, 0.5])  # should be 0


## 2. Non-negativity (Gibbs)

Theorem (Gibbs): $D_{\mathrm{KL}}(p\|q) \ge 0$, with equality iff $p = q$. Numerically:


In [ ]:
examples = [
    ([0.5, 0.5], [0.5, 0.5]),
    ([0.9, 0.1], [0.5, 0.5]),
    ([0.1, 0.9], [0.5, 0.5]),
    ([0.25, 0.25, 0.25, 0.25], [0.7, 0.1, 0.1, 0.1]),
]
for p, q in examples:
    print(f'D_KL = {kl_discrete(p, q):.4f}   p={p}  q={q}')
print('All non-negative?', all(kl_discrete(p, q) >= -1e-12 for p, q in examples))


## 3. Asymmetry

$D_{\mathrm{KL}}(p\|q) \ne D_{\mathrm{KL}}(q\|p)$ in general. The first penalizes $p$-mass that $q$ misses; the second penalizes $q$-mass that $p$ misses. This asymmetry drives the *mode-seeking vs mean-covering* distinction in variational inference.


In [ ]:
p, q = [0.1, 0.9], [0.5, 0.5]
print(f'D_KL(p||q) = {kl_discrete(p, q):.4f}')
print(f'D_KL(q||p) = {kl_discrete(q, p):.4f}')


## 4. Triangle inequality fails

If KL were a metric we would have $D(p\|r) \le D(p\|q) + D(q\|r)$. Pick $p \approx \delta_0$, $q$ uniform, $r \approx \delta_1$:


In [ ]:
p  = [0.99, 0.01]
q  = [0.50, 0.50]
r  = [0.01, 0.99]
lhs = kl_discrete(p, r)
rhs = kl_discrete(p, q) + kl_discrete(q, r)
print(f'D_KL(p||r)               = {lhs:.4f}')
print(f'D_KL(p||q) + D_KL(q||r)  = {rhs:.4f}')
print(f'Triangle inequality violated? {lhs > rhs}')


## 5. Closed-form KL between two Gaussians

For $p = \mathcal{N}(\mu_1, \sigma_1^2)$, $q = \mathcal{N}(\mu_2, \sigma_2^2)$:
$$D_{\mathrm{KL}}(p\|q) = \tfrac{1}{2}\!\left(\frac{\sigma_1^2}{\sigma_2^2} - 1 - \log\frac{\sigma_1^2}{\sigma_2^2} + \frac{(\mu_1 - \mu_2)^2}{\sigma_2^2}\right).$$
We verify against a Monte-Carlo estimate.


In [ ]:
def kl_gaussian_closed(mu1, s1, mu2, s2):
    v1, v2 = s1*s1, s2*s2
    return 0.5 * (v1/v2 - 1.0 - log(v1/v2) + (mu1 - mu2)**2 / v2)

def log_normal_pdf(x, mu, s):
    return -0.5 * log(2.0 * pi * s * s) - (x - mu)**2 / (2.0 * s * s)

def kl_gaussian_mc(mu1, s1, mu2, s2, n=200_000, seed=0):
    rng = random.Random(seed)
    acc = 0.0
    for _ in range(n):
        x = rng.gauss(mu1, s1)
        acc += log_normal_pdf(x, mu1, s1) - log_normal_pdf(x, mu2, s2)
    return acc / n

mu1, s1, mu2, s2 = 0.3, 1.2, -0.5, 2.0
closed = kl_gaussian_closed(mu1, s1, mu2, s2)
mc     = kl_gaussian_mc(mu1, s1, mu2, s2, n=200_000, seed=42)
print(f'closed form = {closed:.4f}')
print(f'monte carlo = {mc:.4f}')
print(f'|diff|      = {abs(closed - mc):.4f}')


## 6. Takeaways

- KL is non-negative, zero iff $p = q$ (Gibbs, via Jensen on $-\log$).
- KL is **asymmetric** and **not** a metric — no triangle inequality, no symmetry.
- The Gaussian closed form is the workhorse of VAEs; the asymmetry powers the mode-seeking vs mean-covering distinction in variational inference.
- In RLHF / GRPO / ReasonMaxxer (Akgül 2025), a KL term anchors the trained policy to a reference, preventing collapse.

**Next:** mutual information $I(X;Y) = D_{\mathrm{KL}}(p(x,y)\|p(x)p(y))$ — concept 39.
